In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [3]:
class Head(nn.Module):
    def __init__(self, d_model: int, head_dim: int):
        super().__init__()
        self.d_model = d_model
        self.q_proj = nn.Linear(d_model, head_dim)
    
    def forward(self, x: torch.Tensor, k: torch.Tensor, v: torch.Tensor):
        q = self.q_proj(x) # (B, T, d_model) @ (d_model, head_dim) = (B, T, head_dim)
        attention = (q @ k.transpose(-2, -1)) # (B, T, head_dim) @ (B, head_dim, T) = (B, T, T)
        attention = attention * (self.d_model ** -0.5) # (B, T, T)
        attention = F.softmax(attention, dim=-1)

        out = attention @ v # (B, T, T) @ (B, T, head_dim) = (B, T, head_dim)
        return out



In [6]:
class GroupedQueryAttentionBlock(nn.Module):
    def __init__(self, d_model: int, num_query_heads: int, num_kv_heads: int):
        super().__init__()

        assert (num_query_heads % num_kv_heads) == 0, "Query Heads should be a multiple of KV Heads"
        assert (d_model % num_query_heads) == 0, "Hidden Embedding Dimension should be a multiple of Number of Heads"

        self.d_model = d_model
        self.num_kv_heads = num_kv_heads
        self.num_query_heads = num_query_heads

        self.num_query_heads_per_group = num_query_heads / num_kv_heads
        self.head_dim = d_model // num_query_heads

        self.k_proj = nn.Linear(self.d_model, self.num_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(self.d_model, self.num_kv_heads * self.head_dim)

        self.heads = nn.ModuleList(
            [ Head(self.d_model, self.head_dim) for _ in range(self.num_query_heads)]
        )
        self.o_proj = nn.Linear(self.d_model, self.d_model)
    
    def forward(self, x: torch.Tensor):
        B, T, _ = x.shape

        k = self.k_proj(x) # (B,T,d_model) @ (d_model, num_kv_heads * head_dim) = (B,T,num_kv_heads * head_dim)
        v = self.v_proj(x) # (B,T,d_model) @ (d_model, num_kv_heads * head_dim) = (B,T,num_kv_heads * head_dim)

        k_groups = k.view(B, T, self.num_kv_heads, self.head_dim)
        v_groups = v.view(B, T, self.num_kv_heads, self.head_dim)

        head_outputs = []

        for i, head in enumerate(self.heads):
            group_id = int(i // self.num_query_heads_per_group)

            k_group = k_groups[:, :, group_id, :] # B,T,head_dim
            v_group = v_groups[:, :, group_id, :] # B,T,head_dim

            head_outputs.append(head(x, k_group, v_group)) # B,T,head_dim
        
        out = torch.cat(head_outputs, -1) # B, T, head_dim * num_query_heads = B, T, d_model
        out = self.o_proj(out) # B, T, d_model

        return out

In [7]:
batch_size = 2
seq_len = 16
embed_dim = 512
num_heads = 8

# Create random input tensor
x = torch.randn(batch_size, seq_len, embed_dim)

# # 1. Multi-Head Attention (MHA): 8 groups for 8 heads (1 KV pair per head)
# mha = GroupedQueryAttentionBlock(embed_dim, num_heads, num_kv_groups=8)
# out_mha = mha(x)

# # 2. Multi-Query Attention (MQA): 1 group for all 8 heads (1 shared KV pair)
# mqa = GroupedQueryAttentionBlock(embed_dim, num_heads, num_kv_groups=1)
# out_mqa = mqa(x)

# 3. Grouped-Query Attention (GQA): 4 groups for 8 heads (2 query heads share 1 KV pair)
gqa = GroupedQueryAttentionBlock(embed_dim, num_heads, num_kv_heads=4)
out_gqa = gqa(x)

print(f"Output shape: {out_gqa.shape}") # Expected: (2, 16, 512)

Output shape: torch.Size([2, 16, 512])
